In [245]:
import pandas as pd
movies_df = pd.read_csv("../../data/raw/rotten_tomatoes/rotten_tomatoes_raw.csv")
movies_df.head()

,title,summary,critics score,critics reviews,audience score,audience reviews,Director,Producer,Distributor,Genre,...,Release Date (Streaming),Runtime,Screenwriter,Production Co,Release Date (Theaters),Aspect Ratio,Rating,Box Office (Gross USA),Sound Mix,Rerelease Date (Theaters)
0,Godfather of MMA,"A documentary on world champion kick boxer, le...",NaN,0 Reviews,NaN,0 Ratings,Peter Normandia,Peter Normandia,Indie Rights,"Documentary, Biography, Sports",...,"Sep 21, 2026",1h 17m,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ice Cream Man,Ice Cream Man follows an idyllic summer town d...,25%,124 Reviews,46%,100+ Verified Ratings,Eli Roth,"Eli Roth, Christopher Woodrow, Kate Harrison",Iconic Events,"Horror, Comedy",...,"Sep 22, 2026",1h 26m,"Eli Roth, Noah Belson","Mass Appeal, Dentsu, Metrol Technology, Head G...","Aug 7, 2026, Wide",Digital 2.39:1,NaN,NaN,NaN,NaN
2,Idiots,Two unqualified bozos (Dave Franco and O'Shea ...,63%,76 Reviews,58%,100+ Verified Ratings,Macon Blair,"Alex Orr, Brandon James, Dave Franco, Ford Cor...",Independent Film Company,"Comedy, Crime",...,"Sep 22, 2026",1h 38m,Macon Blair,"Peachtree Media Partners, Rough House Pictures...","Aug 28, 2026, Wide",NaN,R (Sexual Material|Pervasive Language|Drug Con...,$1.1M,NaN,NaN
3,Twiggy,"An intimate, star-studded dive into the birth ...",100%,20 Reviews,NaN,Fewer than 50 Ratings,Sadie Frost,"Simon Jones, Nick Hamson",Cleopatra Entertainment,Documentary,...,"Sep 22, 2026",1h 34m,NaN,Soho Talent,NaN,NaN,NaN,NaN,NaN,NaN
4,The Magic Faraway Tree,Adapted from Enid Blyton's beloved classic nov...,80%,103 Reviews,81%,250+ Verified Ratings,Ben Gregor,"Pippa Harris, Nicolas Brown, Danny Perkins, Ja...",Vertical,"Kids & Family, Fantasy, Adventure, Comedy",...,"Sep 22, 2026",1h 50m,Simon Farnaby,"Elysian Film Group, StudioCanal, Ashland Hill ...","Aug 21, 2026, Wide",Digital 2.39:1,PG (Some Rude Humor|Mild Action|Brief Language),NaN,Dolby Digital,NaN


In [246]:
# show only number of reviews instead of number of reviews + "Reviews" and change it to float Ex : "30 Reviews" ===> 30
movies_df["critic_reviews_count"] = (
    movies_df["critics reviews"]
    .str.replace(" Reviews", "", regex=False)
    .astype(float)
)

In [247]:
# Remove percentage and change it to float  Ex ! "30%" ===> 30

movies_df["critic_score_percentage"] = (
    movies_df["critics score"]
    .str.replace("%","",regex=False)
    .astype(float)
)
movies_df["audience_score_percentage"] = (
    movies_df["audience score"]
    .str.replace("%","",regex=False)
    .astype(float)
)


In [248]:
# split audience reviews into thre columns 
# Example 1: 
# before: audience reviews[4] = 250+ Verified Ratings
# After : is_audience_rating_verified[4] = True (cause Verified exist in review)
#         audience_rating_count[4] = 250 
#         audience_rating_count_operator[4] = ">=" (cause their + in old value means its more than 250)
# Example 2: 
# before: audience reviews[3] = Fewer than 50 Ratings
# After : is_audience_rating_verified[3] = False (cause Verified don't exist in review)
#         audience_rating_count[3] = 50 
#         audience_rating_count_operator[3] = "<" (cause their + in old value means its more than 250)
movies_df["is_audience_rating_verified"] = movies_df["audience reviews"].str.contains(
    "Verified",
    case=False,
    na=False
)
movies_df["audience_rating_count"] = (
    movies_df["audience reviews"]
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")[0]
    .astype("Int64")
)

#  Identify the operator
movies_df["audience_rating_count_operator"] = "exact"

movies_df.loc[
    movies_df["audience reviews"].str.contains(r"\+", na=False),
    "audience_rating_count_operator"
] = ">="

movies_df.loc[
    movies_df["audience reviews"].str.contains("Fewer than", na=False),
    "audience_rating_count_operator"
] = "<"

In [249]:
# organize release date (streaming ) into three columns year month day
movies_df["Release Date (Streaming)"] = pd.to_datetime(movies_df["Release Date (Streaming)"])

movies_df["release_date_streaming_year"] = movies_df["Release Date (Streaming)"].dt.year
movies_df["release_date_streaming_month"] = movies_df["Release Date (Streaming)"].dt.month
movies_df["release_date_streaming_day"] = movies_df["Release Date (Streaming)"].dt.day

In [250]:
# this column have 135 null values so its better to just delete it
movies_df = movies_df.drop(columns=["Rerelease Date (Theaters)"])

In [251]:
# Extract Wide / Limited / Original
movies_df["wide_or_limited"] = movies_df["Release Date (Theaters)"].str.extract(
    r",\s*(Wide|Limited|Original)$"
)

# Keep only the date
movies_df["Release Date (Theaters)"] = movies_df["Release Date (Theaters)"].str.extract(
    r"^(.+?),\s*(?:Wide|Limited|Original)$"
)[0]

# Convert to datetime
movies_df["Release Date (Theaters)"] = pd.to_datetime(
    movies_df["Release Date (Theaters)"]
)
movies_df["release_date_theaters_year"] = movies_df["Release Date (Theaters)"].dt.year
movies_df["release_date_theaters_month"] = movies_df["Release Date (Theaters)"].dt.month
movies_df["release_date_theaters_day"] = movies_df["Release Date (Theaters)"].dt.day

In [252]:
# split runtime into two columns | Ex: 1h30m =====> 1 | 30
movies_df["runtime_hour"] = (
    movies_df["Runtime"]
    .str.extract(r"(\d+)h")[0]
    .astype("Int64")
)

movies_df["runtime_minutes"] = (
    movies_df["Runtime"]
    .str.extract(r"(\d+)m")[0]
    .astype("Int64")
)

In [253]:
# split Aspect Ration column into two columns 
# Example: 
# Before : Aspect Ratio[0] = Digital 2.39:1
# After  : aspect_ratio[0] = Digital	
#          format_ratio [0] =  2.39:1 
movies_df["format_ratio"] = movies_df["Aspect Ratio"].str.extract(
    r"^([A-Za-z]+)"
)

movies_df["aspect_ratio"] = movies_df["Aspect Ratio"].str.extract(
    r"(\d+\.\d+:\d+)"
)[0]



In [254]:
# change box office from string to integer 
# Example: 
# Before : Box Office (Gross USA)[2] = $1.1M
# After  : box_office_in_dollars[2]  =  1100000.0
movies_df["box_office_in_dollars"] = (
    movies_df["Box Office (Gross USA)"]
    .str.replace("$", "", regex=False)
    .str.replace("M", "e6", regex=False)
    .str.replace("K", "e3", regex=False)
    .astype(float)
)

In [255]:
# remove unecessary columns that we already split of processed
movies_df = movies_df.drop(columns=["Box Office (Gross USA)","Aspect Ratio","critics score","critics reviews","audience score","audience reviews","Release Date (Streaming)","Runtime","Release Date (Theaters)"])

In [256]:
# simple renaming and organization
movies_df = movies_df.rename(columns={
    "Director": "director",
    "Producer": "producer",
    "Distributor": "distributor",
    "Genre": "genre",
    "Original Language": "original_language",
    "Screenwriter": "screenwriter",
    "Production Co": "production_co",
    "Rating": "rating",
    "Sound Mix": "sound_Mix",
    "wide_or_limited":"release_type_theater"
})

In [257]:
movies_df.columns

Index(['title', 'summary', 'director', 'producer', 'distributor', 'genre',
       'original_language', 'screenwriter', 'production_co', 'rating',
       'sound_Mix', 'critic_reviews_count', 'critic_score_percentage',
       'audience_score_percentage', 'is_audience_rating_verified',
       'audience_rating_count', 'audience_rating_count_operator',
       'release_date_streaming_year', 'release_date_streaming_month',
       'release_date_streaming_day', 'release_type_theater',
       'release_date_theaters_year', 'release_date_theaters_month',
       'release_date_theaters_day', 'runtime_hour', 'runtime_minutes',
       'format_ratio', 'aspect_ratio', 'box_office_in_dollars'],
      dtype='object')

In [258]:
movies_df.to_csv("../../data/processed/rotten_tomatoes_cleaned.csv", index=False)